# FM study: Colab setup

Puts the frozen-encoder study's files on a fresh Colab runtime and proves they are the right ones.
It runs **no encoder and produces no scores**. The rules are in `docs/research/FM_RUNBOOK.md`.

**Runtime:** Runtime → Change runtime type → **GPU: A100, L4 or T4**, not RTX PRO 6000 / Blackwell.
**Before you start:** `HF_TOKEN` must be in Colab Secrets (key icon) with notebook access on. Run this
notebook under the Google account that owns `octfsl`.

| Step | What it does | What proves it worked |
|---|---|---|
| 1 | Checks the GPU | A supported GPU is attached |
| 2 | Clones the exact code and builds a separate Python environment | Commit matches; `pip check` passes; PyTorch sees the GPU |
| 3 | Unpacks the images from `octfsl/raw`, then disconnects Drive | All 84,630 manifest images exist |
| 4 | Rebuilds the groups, episodes and rosters | Six files match the laptop's byte for byte |
| 5 | Downloads RETFound, DINOv2, MIRAGE and the Inception reference | Every file matches its pinned SHA256 |
| 6 | Prints the final inventory | Copy this output back to the project thread |

Runtime → Run all is fine: any failed check stops the notebook. Files live in `/content` and vanish
when the runtime ends. Rerunning is safe. The outputs here are a **preview, not the input lock**.

In [ ]:
# Settings and helpers
import csv, datetime, hashlib, json, os, pathlib, shutil, subprocess, time, zipfile

COMMIT = "41782db6cf3de39155d21f77c0b88971e6317ae2"  # the code being checked; change only on purpose
REPO = "https://github.com/Malnoad/Few-shot-rare-diseases"
ROOT, ENV = "/content/Few-shot", "/content/fm-env"
PY = f"{ENV}/bin/python"
PREPARED, ASSETS = "/content/prepared", "/content/assets"
LEDGER = "/content/fm_v1/download_ledger.csv"
RAW = "/content/drive/MyDrive/octfsl/raw"
IMAGES = 84630
ARMS = ("RETFound", "DINOv2", "MIRAGE", "Inception")  # three FMs and the reference; BioMedCLIP is fallback only

# Produced on the laptop from the same code and manifest (9 Nov).
EXPECTED_PREPARED = {
    "groups.csv": "6991d8229149aadf4af5ee53f1e2eb3991805d3bf56ebb5b367337a30224c5d0",
    "nine_way_roster.csv": "72f90c00b98810af46d257fe53d047d93f5312552ebaa478aef72d58da81b586",
    "episodes_development.csv": "cd060bddb434b5de476f4fd3550b8b21481ebaf3eb597298f727517968544695",
    "episodes_reporting.csv": "f9e1e3c6a4f2aacd405171837d472190271da2b03dbce79b619a03900b202b8f",
    "timing_inventory.csv": "a6be7a5ec3ba1bbe220674faaef8e51105bc7e4da59e9672d20e4e4562be97ed",
    "extraction_inventory.csv": "5ca665e4828b2d45c7d90cc6363708d9548793de7019d417e57222ee2b76b44d",
}


def sh(command):
    """Run a shell command, streaming its output; any failure stops the notebook."""
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    if process.wait():
        raise RuntimeError(f"FAILED: {command}")


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

In [ ]:
# Step 1. Check the GPU
query = "nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv,noheader"
gpu = subprocess.run(query, shell=True, capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU: Runtime > Change runtime type > GPU (A100, L4 or T4)"
print(gpu.stdout.strip())
assert float(gpu.stdout.split(",")[1]) < 12, "Blackwell GPU: switch to A100, L4 or T4"

In [ ]:
# Step 2. Get the exact code and build the study environment
if not os.path.isdir(ROOT):
    sh(f"git clone -q {REPO} {ROOT}")
sh(f"git -C {ROOT} checkout -q {COMMIT}")
head = subprocess.check_output(["git", "-C", ROOT, "rev-parse", "HEAD"], text=True).strip()
assert head == COMMIT, head
print("code:", head)

# Separate from Colab's own Python, so its packages cannot conflict. --without-pip needs no ensurepip.
if not os.path.exists(PY):
    sh(f"python3 -m venv --without-pip {ENV}")
sh(f"python3 -m pip --python {PY} install -q -r {ROOT}/requirements-fm.txt")
sh(f"python3 -m pip --python {PY} check")
sh(f"""{PY} -c 'import sys, torch, numpy, scipy, PIL, transformers; assert torch.cuda.is_available(), "PyTorch cannot see the GPU"; print("python", sys.version.split()[0], "torch", torch.__version__, "cuda", torch.version.cuda, "numpy", numpy.__version__)'""")

In [ ]:
# Step 3. Unpack the images from Drive, then disconnect Drive
from google.colab import drive


def extract(zip_path, anchor):
    """Copy every file below the folder named `anchor` into data/raw, skipping macOS junk."""
    count = 0
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            parts = [p for p in info.filename.replace("\\", "/").split("/") if p]
            if not parts or info.filename.endswith(("/", "\\")) or "__MACOSX" in parts or parts[-1].startswith("._"):
                continue
            if anchor not in parts[:-1]:
                continue
            target = pathlib.Path(ROOT, "data/raw", *parts[parts.index(anchor):])
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, target.open("wb") as destination:
                shutil.copyfileobj(source, destination)
            count += 1
    print(f"{pathlib.Path(zip_path).name}: {count} files")


with open(f"{ROOT}/data/manifest.csv", newline="", encoding="utf-8") as stream:
    MANIFEST = [row["path"].replace("\\", "/") for row in csv.DictReader(stream)]


def absent():
    return [p for p in MANIFEST if not os.path.isfile(f"{ROOT}/{p}")]


assert len(MANIFEST) == IMAGES, len(MANIFEST)
if absent():
    drive.mount("/content/drive")
    assert os.path.isdir(RAW), "octfsl/raw not found: is this the Google account that owns octfsl?"
    extract(f"{RAW}/oct2017.zip", "OCT2017")
    extract(f"{RAW}/rare.zip", "rare")
    drive.flush_and_unmount()
missing_images = absent()
assert not missing_images, f"{len(missing_images)} manifest images missing; first: {missing_images[0]}"
print(len(MANIFEST), "manifest images present")

In [ ]:
# Step 4. Rebuild the rosters and compare them with the laptop's
sh(f"{PY} {ROOT}/fm_prepare.py screen --root {ROOT} --out {PREPARED}")
sh(f"{PY} {ROOT}/fm_prepare.py freeze --root {ROOT} --out {PREPARED}")
different = [name for name, digest in EXPECTED_PREPARED.items() if sha256(f"{PREPARED}/{name}") != digest]
assert not different, f"Differs from the laptop, stop and report: {different}"
print("All", len(EXPECTED_PREPARED), "files match the laptop byte for byte, so the 5,146 images and rosters agree.")

In [ ]:
# Step 5a. Define the checkpoint downloads (revisions and digests come from fm_extract.py)
import requests

pins = json.loads(subprocess.check_output([PY, "-c", (
    "import json, fm_extract as x; print(json.dumps(dict(repositories=x.REPOSITORIES, revisions=x.REVISIONS, "
    "sources=x.SOURCE_REVISIONS, digests=x.EXPECTED_ASSET_SHA256)))")], cwd=ROOT, text=True))


def hf(arm, name):
    return f"https://huggingface.co/{pins['repositories'][arm]}/resolve/{pins['revisions'][arm]}/{name}"


def github(repo, arm, path):
    return f"https://raw.githubusercontent.com/{repo}/{pins['sources'][arm]}/{path}"


KERAS = "inception_v3_weights_tf_dim_ordering_tf_kernels_notop.h5"
FILES = [  # arm, file name, source, needs the Hugging Face token
    ("RETFound", "RETFound_mae_natureOCT.pth", hf("RETFound", "RETFound_mae_natureOCT.pth"), True),
    ("RETFound", "models_vit.py", github("rmaphoh/RETFound_MAE", "RETFound", "models_vit.py"), False),
    ("DINOv2", "model.safetensors", hf("DINOv2", "model.safetensors"), False),
    ("DINOv2", "config.json", hf("DINOv2", "config.json"), False),
    ("DINOv2", "preprocessor_config.json", hf("DINOv2", "preprocessor_config.json"), False),
    ("MIRAGE", "model.safetensors", hf("MIRAGE", "model.safetensors"), False),
    ("MIRAGE", "config.json", hf("MIRAGE", "config.json"), False),
    ("MIRAGE", "mirage_hf.py", github("j-morano/MIRAGE", "MIRAGE", "hf/mirage_hf.py"), False),
    ("Inception", KERAS, f"https://storage.googleapis.com/tensorflow/keras-applications/inception_v3/{KERAS}", False),
]
assert {(arm, name) for arm, name, _, _ in FILES} == {(arm, name) for arm in ARMS for name in pins["digests"][arm]}


def record(row):
    """Append to the download-cost ledger the protocol asks for (runbook, R4)."""
    path = pathlib.Path(LEDGER)
    path.parent.mkdir(parents=True, exist_ok=True)
    new = not path.exists()
    with path.open("a", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        if new:
            writer.writerow(["arm", "file", "url", "bytes", "seconds", "utc"])
        writer.writerow(row)


def fetch(arm, name, url, token=None):
    """Download one pinned file. Bytes that fail the digest are deleted, never kept."""
    target = pathlib.Path(ASSETS, arm, name)
    expected = pins["digests"][arm][name]
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and sha256(target) == expected:
        print(f"already verified  {arm}/{name}")
        return
    partial = target.with_name(name + ".part")
    digest, size, start = hashlib.sha256(), 0, time.time()
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    with requests.get(url, headers=headers, stream=True, timeout=60) as response:
        response.raise_for_status()
        with partial.open("wb") as stream:
            for chunk in response.iter_content(1 << 20):
                stream.write(chunk)
                digest.update(chunk)
                size += len(chunk)
    seconds = round(time.time() - start, 1)
    if digest.hexdigest() != expected:
        partial.unlink()
        raise ValueError(f"{arm}/{name} does not match its pinned SHA256; nothing was kept")
    partial.replace(target)
    record([arm, name, url, size, seconds, datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")])
    print(f"downloaded {size / 1e6:9.1f} MB in {seconds:7.1f} s  {arm}/{name}")

In [ ]:
# Step 5b. Download. The token is read only for the RETFound weights and is never stored or printed.
from google.colab import userdata

for arm, name, url, gated in FILES:
    fetch(arm, name, url, userdata.get("HF_TOKEN") if gated else None)

In [ ]:
# Step 6. Final inventory: copy this output back to the project thread
for arm in ARMS:
    for name, expected in pins["digests"][arm].items():
        path = pathlib.Path(ASSETS, arm, name)
        assert sha256(path) == expected, f"{arm}/{name}"
        print(f"OK {path.stat().st_size / 1e6:9.1f} MB  {arm}/{name}")
print()
print(open(LEDGER, encoding="utf-8").read() if os.path.exists(LEDGER) else "No new downloads this session.")
print("code:", subprocess.check_output(["git", "-C", ROOT, "rev-parse", "HEAD"], text=True).strip())
print("Setup complete. No encoder has been run and no score exists.")